# Commands 17A and 17B: Sepsis-by-HM-Subtype Interaction

This notebook tests whether the adjusted association between documented sepsis and documented inpatient palliative-care use differs across mutually exclusive HM subtypes. Run all cells to refresh the model.

## Model specification

The DISCWT-weighted logistic model includes sepsis, mutually exclusive HM subtype, all sepsis-by-subtype interaction terms, continuous age, sex, race/ethnicity, payer, income quartile, cancer-excluded Charlson category, hospital region, hospital location/teaching, hospital bed size, and admission year. Lymphoma is the subtype reference. Adjusted probabilities standardize every subtype/sepsis combination over the observed weighted distribution of all other covariates.

Variance uses year-specific `NIS_STRATUM` with discharges as variance units. Since `HOSP_NIS` is unavailable by study decision, inference is a strata-adjusted approximation rather than full NIS hospital-cluster-adjusted inference.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
from src.phase_9_interaction import main
summary = main()

{
  "included_unweighted_records": 994992,
  "excluded_records": 0,
  "iterations": 8,
  "joint_interaction_test": {
    "wald_chi_square": 181.245,
    "degrees_of_freedom": 8,
    "overall_interaction_p_value": "<0.001"
  },
  "variance_note": "DISCWT-weighted interaction model with year-specific NIS_STRATUM linearization and discharge-level variance units; not full NIS hospital-cluster-adjusted inference.",
  "subtype_estimates": [
    {
      "hm_subtype": "Lymphoma",
      "adjusted_probability_no_sepsis_percent": 6.53,
      "adjusted_probability_sepsis_percent": 17.86,
      "adjusted_absolute_difference_pp": 11.33,
      "difference_ci_95_lower_pp": 10.96,
      "difference_ci_95_upper_pp": 11.7,
      "subtype_p_value": "<0.001",
      "overall_interaction_p_value": "<0.001"
    },
    {
      "hm_subtype": "AML",
      "adjusted_probability_no_sepsis_percent": 12.53,
      "adjusted_probability_sepsis_percent": 25.59,
      "adjusted_absolute_difference_pp": 13.05,
      "dif

## Command 17A: Joint interaction test

In [2]:
interaction_test = pd.DataFrame([summary['joint_interaction_test'], {
    'wald_chi_square': '—', 'degrees_of_freedom': '—', 'overall_interaction_p_value': '—'
}])
interaction_test.insert(0, 'measure', ['Sepsis × HM subtype interaction', 'Total'])
interaction_test.columns = ['Measure', 'Wald chi-square', 'Degrees of freedom', 'P-value']
display(interaction_test.style.hide(axis='index'))

Measure,Wald chi-square,Degrees of freedom,P-value
Sepsis × HM subtype interaction,181.245000,8,<0.001
Total,—,—,—


## Command 17B: Subtype-specific adjusted estimates

In [3]:
subtype = pd.read_csv(REPO_ROOT / 'outputs/phase_9/subtype_adjusted_probabilities.csv', keep_default_na=False)
subtype['Difference 95% CI, pp'] = subtype.apply(lambda row: '—' if row['hm_subtype'] == 'Total' else f"{float(row['difference_ci_95_lower_pp']):.2f}–{float(row['difference_ci_95_upper_pp']):.2f}", axis=1)
subtype = subtype[['hm_subtype', 'adjusted_probability_no_sepsis_percent', 'adjusted_probability_sepsis_percent', 'adjusted_absolute_difference_pp', 'Difference 95% CI, pp', 'subtype_p_value', 'overall_interaction_p_value']]
subtype.columns = ['HM subtype', 'Adjusted probability without sepsis, %', 'Adjusted probability with sepsis, %', 'Adjusted difference, pp', 'Difference 95% CI, pp', 'Subtype p-value', 'Overall interaction p-value']
display(subtype.style.hide(axis='index'))

HM subtype,"Adjusted probability without sepsis, %","Adjusted probability with sepsis, %","Adjusted difference, pp","Difference 95% CI, pp",Subtype p-value,Overall interaction p-value
Lymphoma,6.53,17.86,11.33,10.96–11.70,<0.001,<0.001
AML,12.53,25.59,13.05,12.32–13.78,<0.001,
CML,5.41,15.56,10.14,8.92–11.37,<0.001,
CLL/chronic leukemia,4.95,13.19,8.24,7.76–8.71,<0.001,
ALL/unspecified acute leukemia,8.79,22.31,13.52,12.05–14.99,<0.001,
Other leukemia,7.76,17.31,9.55,8.31–10.79,<0.001,
Myeloma/plasma-cell neoplasm,7.31,15.97,8.66,8.18–9.15,<0.001,
MDS,6.03,14.75,8.72,8.21–9.23,<0.001,
MPN,4.66,10.84,6.19,5.81–6.56,<0.001,
Total,—,—,—,—,—,<0.001


## Figure-ready adjusted probabilities

The following long-format table contains each subtype/sepsis combination and its probability confidence interval.

In [4]:
figure_data = pd.read_csv(REPO_ROOT / 'outputs/phase_9/interaction_figure_data.csv')
figure_data.columns = ['HM subtype', 'Sepsis status', 'Adjusted probability, %', '95% CI lower, %', '95% CI upper, %']
figure_data.loc[len(figure_data)] = ['Total', '—', '—', '—', '—']
display(figure_data.style.hide(axis='index'))

HM subtype,Sepsis status,"Adjusted probability, %","95% CI lower, %","95% CI upper, %"
Lymphoma,No documented sepsis,6.530000,6.430000,6.630000
Lymphoma,Documented sepsis,17.860000,17.500000,18.220000
AML,No documented sepsis,12.530000,12.260000,12.800000
AML,Documented sepsis,25.590000,24.910000,26.260000
CML,No documented sepsis,5.410000,5.120000,5.710000
CML,Documented sepsis,15.560000,14.360000,16.750000
CLL/chronic leukemia,No documented sepsis,4.950000,4.820000,5.080000
CLL/chronic leukemia,Documented sepsis,13.190000,12.730000,13.650000
ALL/unspecified acute leukemia,No documented sepsis,8.790000,8.370000,9.210000
ALL/unspecified acute leukemia,Documented sepsis,22.310000,20.890000,23.720000


## Copy/paste-friendly Markdown

In [5]:
def print_markdown(dataframe, title):
    print(f'## {title}\n')
    headers = list(dataframe.columns)
    print('| ' + ' | '.join(headers) + ' |')
    print('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in dataframe.astype(str).itertuples(index=False, name=None):
        print('| ' + ' | '.join(value.replace('|', '\\|') for value in row) + ' |')
    print()
print_markdown(interaction_test, 'Joint interaction test')
print_markdown(subtype, 'Subtype-specific adjusted estimates')

## Joint interaction test

| Measure | Wald chi-square | Degrees of freedom | P-value |
|---|---|---|---|
| Sepsis × HM subtype interaction | 181.245 | 8 | <0.001 |
| Total | — | — | — |

## Subtype-specific adjusted estimates

| HM subtype | Adjusted probability without sepsis, % | Adjusted probability with sepsis, % | Adjusted difference, pp | Difference 95% CI, pp | Subtype p-value | Overall interaction p-value |
|---|---|---|---|---|---|---|
| Lymphoma | 6.53 | 17.86 | 11.33 | 10.96–11.70 | <0.001 | <0.001 |
| AML | 12.53 | 25.59 | 13.05 | 12.32–13.78 | <0.001 |  |
| CML | 5.41 | 15.56 | 10.14 | 8.92–11.37 | <0.001 |  |
| CLL/chronic leukemia | 4.95 | 13.19 | 8.24 | 7.76–8.71 | <0.001 |  |
| ALL/unspecified acute leukemia | 8.79 | 22.31 | 13.52 | 12.05–14.99 | <0.001 |  |
| Other leukemia | 7.76 | 17.31 | 9.55 | 8.31–10.79 | <0.001 |  |
| Myeloma/plasma-cell neoplasm | 7.31 | 15.97 | 8.66 | 8.18–9.15 | <0.001 |  |
| MDS | 6.03 | 14.75 | 8.72 | 8.21–9.23 | <0.001 |  |
| MPN | 4.66 |

## Interpretation

The significant joint interaction indicates that the adjusted association between documented sepsis and documented inpatient palliative-care use differs across HM subtypes. Sepsis is associated with a positive adjusted absolute difference in every subtype, but the magnitude varies. These observational estimates do not establish causality.